In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../') # 允许导入 src 目录下的模块

import pandas as pd
import numpy as np
from src.processing import preprocess_data
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score

# 1. 加载原始数据
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

# 2. 一键特征工程
train_final, test_final = preprocess_data(train, test)

# 3. 准备模型输入
X = train_final.drop(['SalePrice', 'Id'], axis=1)
y = train_final['SalePrice']

nan_cols = X.columns[X.isnull().any()].tolist()
print(f"含有缺失值的列: {nan_cols}")

# 如果有，看看到底有多少个缺失值
if nan_cols:
    print(X[nan_cols].isnull().sum())

# 4. 简单的交叉验证看看效果 (以 Ridge 回归为例)
model = Ridge(alpha=10)
scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')
rmse_scores = np.sqrt(-scores)

print(f"RMSE 平均分: {rmse_scores.mean():.4f}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
含有缺失值的列: []
RMSE 平均分: 0.1397


In [4]:
model_ridge = Ridge(alpha=10)
model_ridge.fit(X, y)

# 2. 对测试集进行预测 (记得 X_test 要去掉 Id)
X_test = test_final.drop(['Id'], axis=1)
# 别忘了要把 log 过的价格还原回去！使用 expm1 (exp(x)-1)
predictions = np.expm1(model_ridge.predict(X_test))

# 3. 构造提交文件
submission = pd.DataFrame({
    "Id": test_final["Id"],
    "SalePrice": predictions
})

# 4. 保存到 submission 文件夹
submission.to_csv('../submissions/ridge_submission.csv', index=False)